In [1]:
from tuutrag.memgraph import MemgraphConnection
import configparser
from pathlib import Path
from tuutrag.data import D
import json 
from tuutrag.types import *



In [2]:
config = configparser.ConfigParser()
config.read("../config.ini")

port = config.get("Memgraph", "port")
frontend_port = config.get("Memgraph", "frontend_port")
host = config.get("Memgraph", "host")

conn = MemgraphConnection(port=port, frontend_port=frontend_port, host=host)

Creating Memgraph UI connection at localhost:3000, view at: http://localhost:3000/
Creating Memgraph Backend connection at localhost:7687, view at: http://localhost:7687/


In [3]:
# Read data from JSON file
data_file: Path = D.processed / "cross_artifact_relations.json"

parts: list[list[str]] = []

with open(data_file, "r", encoding="utf-8") as f:
    try:
        data = json.load(f)

        # Handle both a single object and a list of objects
        if isinstance(data, dict):
            data = [data]

        skipped = 0
        for obj in data:
            for rel in obj.get("relationships", []):
                # rel is already a list of 3 strings
                if isinstance(rel, list):
                    if len(rel) == 3:
                        parts.append([item.strip() for item in rel])
                    else:
                        skipped += 1
                        print(f"Skipping invalid relationship: {rel}")
                # rel is a comma-separated string
                elif isinstance(rel, str):
                    part = [item.strip() for item in rel.split(",", 2)]
                    if len(part) == 3:
                        parts.append(part)
                    else:
                        skipped += 1
                        print(f"Skipping invalid relationship: {rel}")
                else:
                    skipped += 1
                    print(f"Skipping invalid relationship: {rel}")

    except Exception as e:
        print(f"Error reading {data_file}: {e}")

    print(f"Extracted {len(parts)} relationships")
    print(f"Skipped {skipped} relationships due to invalid format")

Skipping invalid relationship: Facet2, describes type of data in Primary_Result_Summary
Skipping invalid relationship: Facet1, provides independent sub-categorization from Facet2
Skipping invalid relationship: Factet2, provides independent sub-categorization from Facet1
Skipping invalid relationship: NASA PDS, includes classes and enumerations for comprehensive data model within planetary science data management
Skipping invalid relationship: <>
Skipping invalid relationship: Incorrect Service Type Parameter causes, Inconsistent Service Type
Skipping invalid relationship: If Unbind-Reason is End, Provider may terminate service instance and release resources
Skipping invalid relationship: If Unbind-Reason is not End, Initiator may attempt to re-bind before service instance ends
Skipping invalid relationship: Generation 2, identified by version number 1 in BIND operation
Skipping invalid relationship: Generation 3, identified by version number 4 in BIND operation
Skipping invalid relatio

In [4]:
# Upsert data into Memgraph
conn.upsert_universal(parts)

  Batch 1: 5,000 / 212,539 upserted
  Batch 2: 10,000 / 212,539 upserted
  Batch 3: 15,000 / 212,539 upserted
  Batch 4: 20,000 / 212,539 upserted
  Batch 5: 25,000 / 212,539 upserted
  Batch 6: 30,000 / 212,539 upserted
  Batch 7: 35,000 / 212,539 upserted
  Batch 8: 40,000 / 212,539 upserted
  Batch 9: 45,000 / 212,539 upserted
  Batch 10: 50,000 / 212,539 upserted
  Batch 11: 55,000 / 212,539 upserted
  Batch 12: 60,000 / 212,539 upserted
  Batch 13: 65,000 / 212,539 upserted
  Batch 14: 70,000 / 212,539 upserted
  Batch 15: 75,000 / 212,539 upserted
  Batch 16: 80,000 / 212,539 upserted
  Batch 17: 85,000 / 212,539 upserted
  Batch 18: 90,000 / 212,539 upserted
  Batch 19: 95,000 / 212,539 upserted
  Batch 20: 100,000 / 212,539 upserted
  Batch 21: 105,000 / 212,539 upserted
  Batch 22: 110,000 / 212,539 upserted
  Batch 23: 115,000 / 212,539 upserted
  Batch 24: 120,000 / 212,539 upserted
  Batch 25: 125,000 / 212,539 upserted
  Batch 26: 130,000 / 212,539 upserted
  Batch 27: 135

Query: "MATCH (n:Entity)-[r:RELATIONSHIP]->(m:Entity)
RETURN n, r, m"

In [ ]:
string = "MAP ID, must be in range, 0,1,2,3"

parts = [item.strip() for item in string.split(",", 2)]

print(len(parts))